In [ ]:
import os
import cv2
import random

input_root = r"C:\Users\Karim\Contacts\Desktop\split_data\train"
output_root = r"C:\Users\Karim\Contacts\Desktop\split_data\train_AUGMENTED"

rotation_range = (-40, 40)
scale_range = 0.2


def augment_image(img):
    h, w = img.shape[:2]

    # rotation
    angle = random.uniform(*rotation_range)
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1)
    img = cv2.warpAffine(img, M, (w, h))

    # scaling
    scale = random.uniform(1 - scale_range, 1 + scale_range)
    img = cv2.resize(img, None, fx=scale, fy=scale)
    img = cv2.resize(img, (w, h))

    # flips
    if random.random() < 0.5:
        img = cv2.flip(img, 1)
    if random.random() < 0.5:
        img = cv2.flip(img, 0)

    return img


# =========================
# Step 1: load classes
# =========================
class_names = [
    d for d in os.listdir(input_root)
    if os.path.isdir(os.path.join(input_root, d))
]

# =========================
# Step 2: count images per class
# =========================
class_images = {}

for c in class_names:
    folder = os.path.join(input_root, c)
    imgs = [
        f for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".png", ".jpeg"))
    ]
    class_images[c] = imgs

# find max class size
max_size = max(len(v) for v in class_images.values())

print("Class distribution:")
for k, v in class_images.items():
    print(k, len(v))

print("Target size per class:", max_size)

# =========================
# Step 3: balance dataset
# =========================
for class_name, images in class_images.items():

    input_folder = os.path.join(input_root, class_name)
    output_folder = os.path.join(output_root, class_name)

    os.makedirs(output_folder, exist_ok=True)

    # copy all original images first
    for img_name in images:
        img_path = os.path.join(input_folder, img_name)
        img = cv2.imread(img_path)

        if img is None:
            continue

        cv2.imwrite(os.path.join(output_folder, img_name), img)

    current_count = len(images)
    needed = max_size - current_count

    print(f"{class_name}: need {needed} augmented images")

    # augment until balance achieved
    i = 0
    while i < needed:

        img_name = random.choice(images)
        img_path = os.path.join(input_folder, img_name)
        img = cv2.imread(img_path)

        if img is None:
            continue

        aug_img = augment_image(img)

        name, ext = os.path.splitext(img_name)
        new_name = f"{name}_aug_{i}.{ext}"

        cv2.imwrite(os.path.join(output_folder, new_name), aug_img)

        i += 1

print("Dataset balanced successfully!")